
# Linear Regression: Machine Learning Implementation

## 1. Import Required Libraries

## 2. Load the Boston Housing Dataset

## 3. Understand the Dataset
### 3.1 Dataset Description
### 3.2 Display Features and Target
### 3.3 Dataset Shape and Information
### 3.4 Summary Statistics
### 3.5 Check Missing Values

## 4. Exploratory Data Analysis (EDA)
### 4.1 Correlation Analysis
### 4.2 Pair Plot
### 4.3 Feature vs. House Price
### 4.4 Regression Plots

## 5. Prepare the Dataset
### 5.1 Separate Independent Features (X)
### 5.2 Separate Dependent Target (y)

## 6. Split the Dataset
### 6.1 Training Set
### 6.2 Testing Set

## 7. Feature Scaling
### 7.1 Fit the Scaler on Training Data
### 7.2 Transform Training and Testing Data
### 7.3 Save the Scaler

## 8. Linear Regression Model
### 8.1 Initialize the Model
### 8.2 Train the Model
### 8.3 Analyze Coefficients and Intercept
### 8.4 Make Predictions
### 8.5 Compare Actual and Predicted Prices

## 9. Evaluate Linear Regression
### 9.1 R² Score
### 9.2 Actual vs. Predicted Plot
### 9.3 Residual Analysis
### 9.4 MAE, MSE, and RMSE
### 9.5 Adjusted R² Score

## 10. Random Forest Regression
### 10.1 Initialize and Train the Model
### 10.2 Make Predictions
### 10.3 Evaluate the Model
### 10.4 Compare Actual and Predicted Prices

## 11. Compare Linear Regression and Random Forest

## 12. Predict House Prices Using New Data

## 13. Save and Load Trained Models
### 13.1 Save the Linear Regression Model
### 13.2 Save the Random Forest Model
### 13.3 Load and Test Saved Models

## 14. Conclusion


In [ ]:
# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

%matplotlib inline

In [ ]:
# ============================================================
# 2. LOAD THE DATASET
# ============================================================

dataset = pd.read_csv("BostonHousing.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", dataset.shape)

display(dataset.head())

In [ ]:
# ============================================================
# 3. UNDERSTAND THE DATASET
# ============================================================

print("\nDataset Information:")
dataset.info()

print("\nSummary Statistics:")
display(dataset.describe())

print("\nMissing Values:")
print(dataset.isnull().sum())

# Stop if the dataset contains missing values.
if dataset.isnull().sum().any():
    raise ValueError("Please handle missing values before training.")

# Check that the target column exists.
if "medv" not in dataset.columns:
    raise ValueError("The dataset must contain a 'medv' target column.")

In [ ]:
# ============================================================
# 4. EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================

# 4.1 Correlation Matrix
plt.figure(figsize=(12, 8))
sns.heatmap(dataset.corr(numeric_only=True),
            annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
# 4.2 Feature vs. House Price
plt.figure(figsize=(7, 5))
sns.regplot(data=dataset, x="rm", y="medv")
plt.xlabel("Average Number of Rooms")
plt.ylabel("House Price")
plt.title("Rooms vs. House Price")
plt.show()

In [ ]:
# 4.3 Pair Plot for Selected Features
selected_columns = ["rm", "lstat", "ptratio", "medv"]

sns.pairplot(dataset[selected_columns])
plt.show()

In [ ]:
# ============================================================
# 5. SEPARATE FEATURES (X) AND TARGET (y)
# ============================================================

# X = independent variables (inputs)
# y = dependent variable (house price)

X = dataset.drop(columns=["medv"])
y = dataset["medv"]

print("Independent features (X):")
display(X.head())

print("Dependent target (y):")
display(y.head())

In [ ]:
# ============================================================
# 6. SPLIT THE DATASET
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42
)

print("Original dataset:", dataset.shape)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
# ============================================================
# 7. FEATURE SCALING FOR LINEAR REGRESSION
# ============================================================

scaler = StandardScaler()

# Fit only on training data to avoid data leakage.
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data scaled:", X_train_scaled.shape)
print("Testing data scaled:", X_test_scaled.shape)


In [ ]:
# ============================================================
# 8. LINEAR REGRESSION MODEL
# ============================================================

# 8.1 Initialize the model
linear_model = LinearRegression()

In [ ]:
# 8.2 Train the model
linear_model.fit(X_train_scaled, y_train)

print("Linear Regression training completed!")

In [ ]:
# 8.3 Display coefficients and intercept
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": linear_model.coef_
})

print("\nModel Coefficients:")
display(coefficients)

print("Intercept:", linear_model.intercept_)

In [ ]:
# ============================================================
# 9. LINEAR REGRESSION PREDICTIONS
# ============================================================

linear_predictions = linear_model.predict(X_test_scaled)

linear_results = pd.DataFrame({
    "Actual Price": y_test.to_numpy(),
    "Predicted Price": linear_predictions
})

linear_results["Residual"] = (
    linear_results["Actual Price"]
    - linear_results["Predicted Price"]
)

linear_results["Absolute Error"] = (
    linear_results["Residual"].abs()
)

display(linear_results.head(10))


In [ ]:
# ============================================================
# 10. EVALUATE LINEAR REGRESSION
# ============================================================

def evaluate_model(model_name, actual, predicted, feature_count):
    mae = mean_absolute_error(actual, predicted)
    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)
    r2 = r2_score(actual, predicted)

    n = len(actual)

    # Adjusted R² is undefined when n <= feature_count + 1.
    adjusted_r2 = (
        1 - (1 - r2) * (n - 1) / (n - feature_count - 1)
        if n > feature_count + 1
        else np.nan
    )

    print(f"\n{model_name} Evaluation")
    print("-" * 35)
    print(f"MAE:        {mae:.4f}")
    print(f"MSE:        {mse:.4f}")
    print(f"RMSE:       {rmse:.4f}")
    print(f"R² Score:   {r2:.4f}")
    print(f"Adjusted R²: {adjusted_r2:.4f}")

    return {
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R²": r2,
        "Adjusted R²": adjusted_r2
    }


linear_metrics = evaluate_model(
    "Linear Regression",
    y_test,
    linear_predictions,
    X_train.shape[1]
)

In [ ]:
# ============================================================
# 11. ACTUAL VS. PREDICTED PLOT
# ============================================================

plt.figure(figsize=(8, 6))

plt.scatter(y_test, linear_predictions, alpha=0.7)

minimum = min(y_test.min(), linear_predictions.min())
maximum = max(y_test.max(), linear_predictions.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    "r--",
    label="Perfect Predictions"
)

plt.xlabel("Actual House Prices")
plt.ylabel("Predicted House Prices")
plt.title("Linear Regression: Actual vs. Predicted")
plt.legend()
plt.show()


In [ ]:
# ============================================================
# 12. RESIDUAL ANALYSIS
# ============================================================

residuals = y_test.to_numpy() - linear_predictions

# 12.1 Distribution of residuals
sns.histplot(residuals, kde=True)
plt.xlabel("Residuals")
plt.title("Distribution of Linear Regression Residuals")
plt.show()

In [ ]:
# 12.2 Residuals vs. predicted values
plt.figure(figsize=(8, 6))

plt.scatter(linear_predictions, residuals, alpha=0.7)
plt.axhline(y=0, color="red", linestyle="--")

plt.xlabel("Predicted House Prices")
plt.ylabel("Residuals")
plt.title("Residuals vs. Predicted Prices")
plt.show()

In [ ]:
# ============================================================
# 13. RANDOM FOREST REGRESSION
# ============================================================

# Random Forest can use the original, unscaled features.
random_forest_model = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=42
)

# Train the model
random_forest_model.fit(X_train, y_train)

# Make predictions
rf_predictions = random_forest_model.predict(X_test)

print("Random Forest training completed!")

In [ ]:
# ============================================================
# 14. EVALUATE RANDOM FOREST
# ============================================================

rf_metrics = evaluate_model(
    "Random Forest",
    y_test,
    rf_predictions,
    X_train.shape[1]
)

rf_results = pd.DataFrame({
    "Actual Price": y_test.to_numpy(),
    "Predicted Price": rf_predictions
})

rf_results["Residual"] = (
    rf_results["Actual Price"]
    - rf_results["Predicted Price"]
)

display(rf_results.head(10))


In [ ]:
# ============================================================
# 15. COMPARE BOTH MODELS
# ============================================================

comparison = pd.DataFrame([
    linear_metrics,
    rf_metrics
])

print("Model Performance Comparison:")
display(comparison.round(4))

# Lower RMSE indicates smaller prediction errors.
best_model = comparison.loc[comparison["RMSE"].idxmin(), "Model"]

print("Model with the lower test RMSE:", best_model)

In [ ]:
# ============================================================
# 16. PREDICT A NEW HOUSE PRICE
# ============================================================

# Example: predict the first house in the dataset.
# This is an illustration, not a genuinely new observation.

new_house = X.iloc[[0]]

# Linear Regression requires scaled features.
new_house_scaled = scaler.transform(new_house)

new_linear_prediction = linear_model.predict(new_house_scaled)[0]

# Random Forest uses original features.
new_rf_prediction = random_forest_model.predict(new_house)[0]

print("Linear Regression predicted price:",
      round(new_linear_prediction, 4))

print("Random Forest predicted price:",
      round(new_rf_prediction, 4))

In [ ]:
# ============================================================
# 17. SAVE THE MODELS AND SCALER
# ============================================================

with open("linear_regression_model.pkl", "wb") as file:
    pickle.dump(linear_model, file)

with open("random_forest_model.pkl", "wb") as file:
    pickle.dump(random_forest_model, file)

with open("scaling.pkl", "wb") as file:
    pickle.dump(scaler, file)

print("Models and scaler saved successfully!")


In [ ]:
# ============================================================
# 18. LOAD AND TEST THE SAVED MODELS
# ============================================================

with open("linear_regression_model.pkl", "rb") as file:
    loaded_linear_model = pickle.load(file)

with open("random_forest_model.pkl", "rb") as file:
    loaded_rf_model = pickle.load(file)

with open("scaling.pkl", "rb") as file:
    loaded_scaler = pickle.load(file)

# Predict using the loaded Linear Regression model.
loaded_linear_prediction = loaded_linear_model.predict(
    loaded_scaler.transform(new_house)
)[0]

# Predict using the loaded Random Forest model.
loaded_rf_prediction = loaded_rf_model.predict(new_house)[0]

print("Loaded Linear Regression prediction:",
      round(loaded_linear_prediction, 4))

print("Loaded Random Forest prediction:",
      round(loaded_rf_prediction, 4))

In [ ]:
# ============================================================
# 19. CONCLUSION
# ============================================================

print("\nMachine Learning Workflow Completed!")
print("Models trained, evaluated, compared, and saved.")
